In [1]:
import os
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv

In [2]:
# 1. Configuración de Entorno Profesional
load_dotenv()
project_id = 'mlopsmarketingproject'
dataset_id = 'olist_marketing_qualified_leads_dataset'
location = 'northamerica-northeast1'

client = bigquery.Client(project=project_id)

In [3]:
# 2. Query Maestro (Pushdown Logic)
# Unimos leads con sus cierres y el total de ventas (LTV)
sql_abt = f"""
WITH seller_performance AS (
    SELECT 
        seller_id, 
        SUM(price) as total_revenue,
        COUNT(order_id) as total_orders
    FROM `{project_id}.{dataset_id}.olist_order_items_dataset`
    GROUP BY 1
)
SELECT 
    mql.mql_id,
    mql.first_contact_date,
    mql.landing_page_id,
    mql.origin,
    cd.business_segment,
    cd.lead_type,
    cd.lead_behaviour_profile, -- Perfiles clave: cat, wolf, shark, eagle [5]
    cd.won_date,
    IF(cd.won_date IS NOT NULL, 1, 0) as converted,
    COALESCE(sp.total_revenue, 0) as ltv_revenue,
    COALESCE(sp.total_orders, 0) as orders_count
FROM `{project_id}.{dataset_id}.olist_marketing_qualified_leads_dataset` mql
LEFT JOIN `{project_id}.{dataset_id}.olist_closed_deals_dataset` cd 
    ON mql.mql_id = cd.mql_id
LEFT JOIN seller_performance sp 
    ON cd.seller_id = sp.seller_id
"""

In [4]:
print("⏳ Generando ABT en BigQuery y descargando para ML...")
df_abt = client.query(sql_abt, location=location).to_dataframe()

# 3. Validación de Calidad Inmediata
print(f"✅ ABT Generada. Dimensiones: {df_abt.shape}")
print(df_abt.info())

⏳ Generando ABT en BigQuery y descargando para ML...


c:\Users\User\Desktop\Software y Clases\BigData\OList\olist-project-sa\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ ABT Generada. Dimensiones: (8000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   mql_id                  8000 non-null   object             
 1   first_contact_date      8000 non-null   dbdate             
 2   landing_page_id         8000 non-null   object             
 3   origin                  7940 non-null   object             
 4   business_segment        841 non-null    object             
 5   lead_type               836 non-null    object             
 6   lead_behaviour_profile  665 non-null    object             
 7   won_date                842 non-null    datetime64[us, UTC]
 8   converted               8000 non-null   Int64              
 9   ltv_revenue             8000 non-null   float64            
 10  orders_count            8000 non-null   Int64              
dtypes: 

# Nueva Sección

In [6]:
df_abt['converted'] = df_abt['won_date'].notna().astype(int)

# Columnas críticas de calificación
qualification_cols = ['lead_type', 'lead_behaviour_profile', 'business_segment']

# Imputamos con un valor que tenga sentido de negocio
df_abt[qualification_cols] = df_abt[qualification_cols].fillna('not_qualified')

In [7]:
print(display(df_abt.head()))

,mql_id,first_contact_date,landing_page_id,origin,business_segment,lead_type,lead_behaviour_profile,won_date,converted,ltv_revenue,orders_count
0,6710565534bb35faf7fad5971cf885b3,2018-02-02,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
1,4da6c4d07033d355453ce49273d591c6,2018-05-18,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
2,d788c46ad6a6c020b5062c1a99f55b2c,2018-05-15,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
3,9e4903ea29841bc16eab71500292e9b0,2018-05-21,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
4,e8432fb72c61c9066957124e5a420a05,2017-10-09,1722481ac9e5371e5099dea226b5421d,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0


None


In [8]:
# Guardamos la ABT como una tabla nueva en BigQuery
destination_table = f"{project_id}.{dataset_id}.abt_marketing_leads"

df_abt.to_gbq(
    destination_table=destination_table,
    project_id=project_id,
    if_exists='replace', # O 'append' si fuera un proceso diario
    location=location
)
print(f"✅ ABT persistida en BigQuery: {destination_table}")

C:\Users\User\AppData\Local\Temp\ipykernel_20464\4155827150.py:4: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_abt.to_gbq(


✅ ABT persistida en BigQuery: mlopsmarketingproject.olist_marketing_qualified_leads_dataset.abt_marketing_leads


In [11]:
# Parquet preserva los tipos (datetimes, ints, floats) y comprime el tamaño
df_abt.to_parquet('C:\\Users\\User\\Desktop\\Software y Clases\\BigData\\OList\\olist-project-sa\\Data\\Processed\\abt_marketing.parquet', index=False)